In [0]:
%sql
use catalog misgauravcatalog;
use schema default;

In [0]:
%sql
create table orders_ext(
order_id int,
order_date string,
customer_id int,
order_status string)
using delta
LOCATION 'gs://databricksfolder/deltatable/'
TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
%sql
show tables;

In [0]:
%sql
DESCRIBE DETAIL misgauravcatalog.default.orders_ext;
-- format	delta
-- id	a8db3ff1-87cf-4518-9989-824a9c6496ed
-- name	misgauravcatalog.default.orders_ext
-- description	null
-- location	gs://databricksfolder/deltatable
-- createdAt	2026-06-06T04:10:38.831Z
-- lastModified	2026-06-06T04:10:42.822Z
-- partitionColumns	[]
-- clusteringColumns	[]
-- numFiles	0
-- sizeInBytes	0
-- properties	{"delta.enableChangeDataFeed":"true","delta.enableDeletionVectors":"true"}
-- minReaderVersion	3
-- minWriterVersion	7
-- tableFeatures	["appendOnly","changeDataFeed","deletionVectors","invariants"]
-- statistics	{"numRowsDeletedByDeletionVectors":0,"numDeletionVectors":0}
-- clusterByAuto	FALSE

In [0]:
%fs
ls gs://databricksfolder/deltatable
path,name,size,modificationTime
gs://databricksfolder/deltatable/_delta_log/,_delta_log/,0,0

In [0]:
%sql
describe history misgauravcatalog.default.orders_ext;
-- version	0
-- timestamp	2026-06-06T04:10:42.822Z
-- userId	218129261881966
-- userName	gauravmishra010@gmail.com
-- operation	CREATE TABLE
-- operationParameters	{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null
-- notebook	{"notebookId":"2268278810704620"}
-- queryHistoryStatementId	ada2fa9d-702c-4a99-b822-d42b971650ae
-- clusterId	0606-025702-2u88m6yc-v2n
-- readVersion	null
-- isolationLevel	WriteSerializable
-- isBlindAppend	TRUE
-- operationMetrics	{}
-- userMetadata	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13

In [0]:
%sql
insert into orders_ext values 
(1,	'2013-07-25 00:00:00.0', 11599,	'CLOSED'),
(2,	'2013-07-25 00:00:00.0', 256,	'PENDING_PAYMENT'),
(3,	'2013-07-25 00:00:00.0', 12111,	'COMPLETE'),
(4,	'2013-07-25 00:00:00.0', 8827,	'CLOSED'),
(5,	'2013-07-25 00:00:00.0', 11318,	'COMPLETE');

In [0]:
%fs
ls gs://databricksfolder/deltatable
path,name,size,modificationTime
gs://databricksfolder/deltatable/_delta_log/,_delta_log/,0,0
gs://databricksfolder/deltatable/part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet,part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet,1508,1780719183160

In [0]:
%sql
describe history misgauravcatalog.default.orders_ext;
-- version	1	0
-- timestamp	2026-06-06T04:13:06.222Z	2026-06-06T04:10:42.822Z
-- userId	218129261881966	218129261881966
-- userName	gauravmishra010@gmail.com	gauravmishra010@gmail.com
-- operation	WRITE	CREATE TABLE
-- operationParameters	{"mode":"Append","statsOnLoad":"false","partitionBy":"[]"}	{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null	null
-- notebook	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}
-- queryHistoryStatementId	12445f3d-213b-43e4-9d65-363eb4b4e6fe	ada2fa9d-702c-4a99-b822-d42b971650ae
-- clusterId	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n
-- readVersion	0	null
-- isolationLevel	WriteSerializable	WriteSerializable
-- isBlindAppend	TRUE	TRUE
-- operationMetrics	{"numFiles":"1","numOutputRows":"5","numOutputBytes":"1508"}	{}
-- userMetadata	null	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
%sql
select * from table_changes('orders_ext', 1); -- 1 means from which version we want to see the changes.
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 1,2013-07-25 00:00:00.0,11599,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 5,2013-07-25 00:00:00.0,11318,COMPLETE,insert,1,2026-06-06T04:13:06.222Z


In [0]:
# till now, I mean insert operation, No _change_data folder has been created. 

In [0]:
%fs
ls gs://databricksfolder/deltatable
path,name,size,modificationTime
gs://databricksfolder/deltatable/_delta_log/,_delta_log/,0,0
gs://databricksfolder/deltatable/part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet,part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet,1508,1780719183160


In [0]:
%sql
delete from orders_ext where order_id = 3;
-- delete commands will create 2 files -> 1. deletion vector file and 2. parquet file
-- num_affected_rows
-- 1


In [0]:
dbutils.fs.ls('gs://databricksfolder/deltatable')
# [FileInfo(path='gs://databricksfolder/deltatable/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/deltatable/deletion_vector_584c5eb3-62c9-4329-b5d9-1ba17dc4f174.bin', name='deletion_vector_584c5eb3-62c9-4329-b5d9-1ba17dc4f174.bin', size=43, modificationTime=1780719474725),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet', name='part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet', size=1498, modificationTime=1780719481672),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet', name='part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet', size=1508, modificationTime=1780719183160)]

In [0]:
%sql
describe history misgauravcatalog.default.orders_ext;
-- version	3	2	1	0
-- timestamp	2026-06-06T04:18:04.710Z	2026-06-06T04:17:58.092Z	2026-06-06T04:13:06.222Z	2026-06-06T04:10:42.822Z
-- userId	218129261881966	218129261881966	218129261881966	218129261881966
-- userName	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com
-- operation	OPTIMIZE	DELETE	WRITE	CREATE TABLE
-- operationParameters	{"predicate":"[]","auto":"true","clusterBy":"[]","zOrderBy":"[]","batchId":"0"}	{"predicate":"[\"(order_id#14302 = 3)\"]"}	{"mode":"Append","statsOnLoad":"false","partitionBy":"[]"}	{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null	null	null	null
-- notebook	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}
-- queryHistoryStatementId	41c64dff-8905-4a00-b728-4494ad0ddae1	41c64dff-8905-4a00-b728-4494ad0ddae1	12445f3d-213b-43e4-9d65-363eb4b4e6fe	ada2fa9d-702c-4a99-b822-d42b971650ae
-- clusterId	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n
-- readVersion	2	1	0	null
-- isolationLevel	SnapshotIsolation	WriteSerializable	WriteSerializable	WriteSerializable
-- isBlindAppend	FALSE	FALSE	TRUE	TRUE
-- operationMetrics	{"numRemovedFiles":"1","numRemovedBytes":"1508","p25FileSize":"1498","numDeletionVectorsRemoved":"1","minFileSize":"1498","numAddedFiles":"1","maxFileSize":"1498","p75FileSize":"1498","p50FileSize":"1498","numAddedBytes":"1498"}	{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"0","executionTimeMs":"3181","numDeletionVectorsUpdated":"0","numDeletedRows":"1","scanTimeMs":"2767","numAddedFiles":"0","numAddedBytes":"0","rewriteTimeMs":"414"}	{"numFiles":"1","numOutputRows":"5","numOutputBytes":"1508"}	{}
-- userMetadata	null	null	null	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13

In [0]:
%sql
select * from table_changes('orders_ext', 1);
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 1,2013-07-25 00:00:00.0,11599,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 5,2013-07-25 00:00:00.0,11318,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,delete,2,2026-06-06T04:17:58.092Z


In [0]:
%sql
select * from table_changes('orders_ext', 2);
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,delete,2,2026-06-06T04:17:58.092Z


In [0]:
# you can't read single delta file by using spark method: 
# gs://databricksfolder/deltatable/part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet

In [0]:
# this will not work 
df = spark.read.format('delta').option('header', True).load('gs://databricksfolder/deltatable/part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet')
# or 
df = spark.read.format('parquet').option('header', True).load('gs://databricksfolder/deltatable/part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet')
df.show()

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS misgauravcatalog.default.misgauravvolume;

In [0]:
dbutils.fs.ls("/Volumes/misgauravcatalog/default/misgauravvolume/")

In [0]:
dbutils.fs.cp("gs://databricksfolder/deltatable/part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet", "/Volumes/misgauravcatalog/default/misgauravvolume/deleted_data/file")

In [0]:
dbutils.fs.ls("/Volumes/misgauravcatalog/default/misgauravvolume/deleted_data")
# [FileInfo(path='dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/deleted_data/file', name='file', size=1498, modificationTime=1780720259724)]

In [0]:
%fs
ls /Volumes/misgauravcatalog/default/misgauravvolume/
path,name,size,modificationTime
dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/deleted_data/,deleted_data/,0,0

In [0]:
df = spark.read.format('parquet').option('header', True).load('dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/deleted_data/file')
df.show()
# +--------+--------------------+-----------+---------------+
# |order_id|          order_date|customer_id|   order_status|
# +--------+--------------------+-----------+---------------+
# |       1|2013-07-25 00:00:...|      11599|         CLOSED|
# |       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
# |       4|2013-07-25 00:00:...|       8827|         CLOSED|
# |       5|2013-07-25 00:00:...|      11318|       COMPLETE|
# +--------+--------------------+-----------+---------------+

In [0]:
%sql
describe history misgauravcatalog.default.orders_ext;
-- version	3	2	1	0
-- timestamp	2026-06-06T04:18:04.710Z	2026-06-06T04:17:58.092Z	2026-06-06T04:13:06.222Z	2026-06-06T04:10:42.822Z
-- userId	218129261881966	218129261881966	218129261881966	218129261881966
-- userName	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com
-- operation	OPTIMIZE	DELETE	WRITE	CREATE TABLE
-- operationParameters	{"predicate":"[]","auto":"true","clusterBy":"[]","zOrderBy":"[]","batchId":"0"}	{"predicate":"[\"(order_id#14302 = 3)\"]"}	{"mode":"Append","statsOnLoad":"false","partitionBy":"[]"}	{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null	null	null	null
-- notebook	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}
-- queryHistoryStatementId	41c64dff-8905-4a00-b728-4494ad0ddae1	41c64dff-8905-4a00-b728-4494ad0ddae1	12445f3d-213b-43e4-9d65-363eb4b4e6fe	ada2fa9d-702c-4a99-b822-d42b971650ae
-- clusterId	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n
-- readVersion	2	1	0	null
-- isolationLevel	SnapshotIsolation	WriteSerializable	WriteSerializable	WriteSerializable
-- isBlindAppend	FALSE	FALSE	TRUE	TRUE
-- operationMetrics	{"numRemovedFiles":"1","numRemovedBytes":"1508","p25FileSize":"1498","numDeletionVectorsRemoved":"1","minFileSize":"1498","numAddedFiles":"1","maxFileSize":"1498","p75FileSize":"1498","p50FileSize":"1498","numAddedBytes":"1498"}	{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"0","executionTimeMs":"3181","numDeletionVectorsUpdated":"0","numDeletedRows":"1","scanTimeMs":"2767","numAddedFiles":"0","numAddedBytes":"0","rewriteTimeMs":"414"}	{"numFiles":"1","numOutputRows":"5","numOutputBytes":"1508"}	{}
-- userMetadata	null	null	null	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13

In [0]:
%sql
select * from table_changes('orders_ext', 1)
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 1,2013-07-25 00:00:00.0,11599,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 5,2013-07-25 00:00:00.0,11318,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,delete,2,2026-06-06T04:17:58.092Z


In [0]:
dbutils.fs.ls('gs://databricksfolder/deltatable')
# [FileInfo(path='gs://databricksfolder/deltatable/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/deltatable/deletion_vector_584c5eb3-62c9-4329-b5d9-1ba17dc4f174.bin', name='deletion_vector_584c5eb3-62c9-4329-b5d9-1ba17dc4f174.bin', size=43, modificationTime=1780719474725),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet', name='part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet', size=1498, modificationTime=1780719481672),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet', name='part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet', size=1508, modificationTime=1780719183160)]

In [0]:
%sql
select * from orders_ext;
-- order_id,order_date,customer_id,order_status
-- 1,2013-07-25 00:00:00.0,11599,CLOSED
-- 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
-- 4,2013-07-25 00:00:00.0,8827,CLOSED
-- 5,2013-07-25 00:00:00.0,11318,COMPLETE


In [0]:
%sql
update orders_ext set order_status = 'value' where order_id = 4; -- now, you'll see _change_data folder
-- update commands will create 2 parquet files, and 1 deletion vector file
-- num_affected_rows
-- 1


In [0]:
dbutils.fs.ls('gs://databricksfolder/deltatable')
# [FileInfo(path='gs://databricksfolder/deltatable/_delta_log/', name='_delta_log/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/deltatable/_change_data/', name='_change_data/', size=0, modificationTime=0),
#  FileInfo(path='gs://databricksfolder/deltatable/deletion_vector_584c5eb3-62c9-4329-b5d9-1ba17dc4f174.bin', name='deletion_vector_584c5eb3-62c9-4329-b5d9-1ba17dc4f174.bin', size=43, modificationTime=1780719474725),
#  FileInfo(path='gs://databricksfolder/deltatable/deletion_vector_a105bb46-17c3-4e4f-8446-808d915ae1e4.bin', name='deletion_vector_a105bb46-17c3-4e4f-8446-808d915ae1e4.bin', size=43, modificationTime=1780720600731),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-0251bbea-526d-42b5-bbe3-b475b50c6f4e.c000.snappy.parquet', name='part-00000-0251bbea-526d-42b5-bbe3-b475b50c6f4e.c000.snappy.parquet', size=1720, modificationTime=1780720603251),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-62b52bbd-6c0b-434b-b890-23633b6267b2.c000.snappy.parquet', name='part-00000-62b52bbd-6c0b-434b-b890-23633b6267b2.c000.snappy.parquet', size=1484, modificationTime=1780720610827),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet', name='part-00000-788233b4-e04d-4caa-9c37-b3aaa7c6333c.c000.snappy.parquet', size=1498, modificationTime=1780719481672),
#  FileInfo(path='gs://databricksfolder/deltatable/part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet', name='part-00000-a38fc5fe-df22-41e0-ae5d-40fd0287915d.c000.snappy.parquet', size=1508, modificationTime=1780719183160)]

In [0]:
dbutils.fs.cp("gs://databricksfolder/deltatable/part-00000-62b52bbd-6c0b-434b-b890-23633b6267b2.c000.snappy.parquet", "/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/file1/file")

In [0]:
dbutils.fs.cp("gs://databricksfolder/deltatable/part-00000-0251bbea-526d-42b5-bbe3-b475b50c6f4e.c000.snappy.parquet", "/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/file2/file")

In [0]:
dbutils.fs.ls("/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/")
# [FileInfo(path='dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/file1/', name='file1/', size=0, modificationTime=0),
#  FileInfo(path='dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/file2/', name='file2/', size=0, modificationTime=0)]

In [0]:
df1 = spark.read.format('parquet').option('header', True).load('dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/file1/*')
df1.show()
# +--------+--------------------+-----------+---------------+
# |order_id|          order_date|customer_id|   order_status|
# +--------+--------------------+-----------+---------------+
# |       1|2013-07-25 00:00:...|      11599|         CLOSED|
# |       2|2013-07-25 00:00:...|        256|PENDING_PAYMENT|
# |       5|2013-07-25 00:00:...|      11318|       COMPLETE|
# |       4|2013-07-25 00:00:...|       8827|          value|
# +--------+--------------------+-----------+---------------+

df2 = spark.read.format('parquet').option('header', True).load('dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/file2/*')
df2.show()
# +--------+--------------------+-----------+------------+------------+--------+
# |order_id|          order_date|customer_id|order_status|_change_type|__is_cdc|
# +--------+--------------------+-----------+------------+------------+--------+
# |       4|2013-07-25 00:00:...|       8827|       value|        NULL|   false|
# +--------+--------------------+-----------+------------+------------+--------+

In [0]:
%sql
describe history misgauravcatalog.default.orders_ext;
-- version	5	4	3	2	1	0
-- timestamp	2026-06-06T04:36:53.837Z	2026-06-06T04:36:47.023Z	2026-06-06T04:18:04.710Z	2026-06-06T04:17:58.092Z	2026-06-06T04:13:06.222Z	2026-06-06T04:10:42.822Z
-- userId	218129261881966	218129261881966	218129261881966	218129261881966	218129261881966	218129261881966
-- userName	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com
-- operation	OPTIMIZE	UPDATE	OPTIMIZE	DELETE	WRITE	CREATE TABLE
-- operationParameters	{"predicate":"[]","auto":"true","clusterBy":"[]","zOrderBy":"[]","batchId":"0"}	{"predicate":"[\"(order_id#15856 = 4)\"]"}	{"predicate":"[]","auto":"true","clusterBy":"[]","zOrderBy":"[]","batchId":"0"}	{"predicate":"[\"(order_id#14302 = 3)\"]"}	{"mode":"Append","statsOnLoad":"false","partitionBy":"[]"}	{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null	null	null	null	null	null
-- notebook	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}
-- queryHistoryStatementId	97d86a60-0f3e-40fb-868b-51f364e85355	97d86a60-0f3e-40fb-868b-51f364e85355	41c64dff-8905-4a00-b728-4494ad0ddae1	41c64dff-8905-4a00-b728-4494ad0ddae1	12445f3d-213b-43e4-9d65-363eb4b4e6fe	ada2fa9d-702c-4a99-b822-d42b971650ae
-- clusterId	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n
-- readVersion	4	3	2	1	0	null
-- isolationLevel	SnapshotIsolation	WriteSerializable	SnapshotIsolation	WriteSerializable	WriteSerializable	WriteSerializable
-- isBlindAppend	FALSE	FALSE	FALSE	FALSE	TRUE	TRUE
-- operationMetrics	{"numRemovedFiles":"2","numRemovedBytes":"3218","p25FileSize":"1484","numDeletionVectorsRemoved":"1","minFileSize":"1484","numAddedFiles":"1","maxFileSize":"1484","p75FileSize":"1484","p50FileSize":"1484","numAddedBytes":"1484"}	{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"1","executionTimeMs":"5499","numDeletionVectorsUpdated":"0","scanTimeMs":"2435","numAddedFiles":"1","numUpdatedRows":"1","numAddedBytes":"1720","rewriteTimeMs":"3062"}	{"numRemovedFiles":"1","numRemovedBytes":"1508","p25FileSize":"1498","numDeletionVectorsRemoved":"1","minFileSize":"1498","numAddedFiles":"1","maxFileSize":"1498","p75FileSize":"1498","p50FileSize":"1498","numAddedBytes":"1498"}	{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"0","executionTimeMs":"3181","numDeletionVectorsUpdated":"0","numDeletedRows":"1","scanTimeMs":"2767","numAddedFiles":"0","numAddedBytes":"0","rewriteTimeMs":"414"}	{"numFiles":"1","numOutputRows":"5","numOutputBytes":"1508"}	{}
-- userMetadata	null	null	null	null	null	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13

In [0]:
%sql
select * from table_changes('orders_ext', 1)
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,update_preimage,4,2026-06-06T04:36:47.023Z
-- 4,2013-07-25 00:00:00.0,8827,value,update_postimage,4,2026-06-06T04:36:47.023Z
-- 1,2013-07-25 00:00:00.0,11599,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 5,2013-07-25 00:00:00.0,11318,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,delete,2,2026-06-06T04:17:58.092Z


In [0]:
dbutils.fs.ls('gs://databricksfolder/deltatable/_change_data/')
# [FileInfo(path='gs://databricksfolder/deltatable/_change_data/cdc-00001-559f7a58-e3b0-46e3-bf57-960763086bbe.c000.snappy.parquet', name='cdc-00001-559f7a58-e3b0-46e3-bf57-960763086bbe.c000.snappy.parquet', size=1874, modificationTime=1780720603251)]

In [0]:
dbutils.fs.cp("gs://databricksfolder/deltatable/_change_data/cdc-00001-559f7a58-e3b0-46e3-bf57-960763086bbe.c000.snappy.parquet", "/Volumes/misgauravcatalog/default/misgauravvolume/cdc_folder_data/file")

In [0]:
dbutils.fs.ls('/Volumes/misgauravcatalog/default/misgauravvolume/')
# [FileInfo(path='dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/cdc_folder_data/', name='cdc_folder_data/', size=0, modificationTime=0),
#  FileInfo(path='dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/deleted_data/', name='deleted_data/', size=0, modificationTime=0),
#  FileInfo(path='dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/updated_data/', name='updated_data/', size=0, modificationTime=0)]

In [0]:
dbutils.fs.ls('/Volumes/misgauravcatalog/default/misgauravvolume/cdc_folder_data')
# [FileInfo(path='dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/cdc_folder_data/file', name='file', size=1874, modificationTime=1780721742655)]

In [0]:
df3 = spark.read.format('parquet').option('header', True).load('dbfs:/Volumes/misgauravcatalog/default/misgauravvolume/cdc_folder_data/file')
df3.show()
# +--------+--------------------+-----------+------------+----------------+--------+
# |order_id|          order_date|customer_id|order_status|    _change_type|__is_cdc|
# +--------+--------------------+-----------+------------+----------------+--------+
# |       4|2013-07-25 00:00:...|       8827|      CLOSED| update_preimage|    true|
# |       4|2013-07-25 00:00:...|       8827|       value|update_postimage|    true|
# +--------+--------------------+-----------+------------+----------------+--------+

In [0]:
%sql
describe history misgauravcatalog.default.orders_ext;
-- version	5	4	3	2	1	0
-- timestamp	2026-06-06T04:36:53.837Z	2026-06-06T04:36:47.023Z	2026-06-06T04:18:04.710Z	2026-06-06T04:17:58.092Z	2026-06-06T04:13:06.222Z	2026-06-06T04:10:42.822Z
-- userId	218129261881966	218129261881966	218129261881966	218129261881966	218129261881966	218129261881966
-- userName	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com	gauravmishra010@gmail.com
-- operation	OPTIMIZE	UPDATE	OPTIMIZE	DELETE	WRITE	CREATE TABLE
-- operationParameters	{"predicate":"[]","auto":"true","clusterBy":"[]","zOrderBy":"[]","batchId":"0"}	{"predicate":"[\"(order_id#15856 = 4)\"]"}	{"predicate":"[]","auto":"true","clusterBy":"[]","zOrderBy":"[]","batchId":"0"}	{"predicate":"[\"(order_id#14302 = 3)\"]"}	{"mode":"Append","statsOnLoad":"false","partitionBy":"[]"}	{"partitionBy":"[]","clusterBy":"[]","description":null,"isManaged":"false","properties":"{\"delta.enableChangeDataFeed\":\"true\",\"delta.enableDeletionVectors\":\"true\"}","statsOnLoad":"false"}
-- job	null	null	null	null	null	null
-- notebook	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}	{"notebookId":"2268278810704620"}
-- queryHistoryStatementId	97d86a60-0f3e-40fb-868b-51f364e85355	97d86a60-0f3e-40fb-868b-51f364e85355	41c64dff-8905-4a00-b728-4494ad0ddae1	41c64dff-8905-4a00-b728-4494ad0ddae1	12445f3d-213b-43e4-9d65-363eb4b4e6fe	ada2fa9d-702c-4a99-b822-d42b971650ae
-- clusterId	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n	0606-025702-2u88m6yc-v2n
-- readVersion	4	3	2	1	0	null
-- isolationLevel	SnapshotIsolation	WriteSerializable	SnapshotIsolation	WriteSerializable	WriteSerializable	WriteSerializable
-- isBlindAppend	FALSE	FALSE	FALSE	FALSE	TRUE	TRUE
-- operationMetrics	{"numRemovedFiles":"2","numRemovedBytes":"3218","p25FileSize":"1484","numDeletionVectorsRemoved":"1","minFileSize":"1484","numAddedFiles":"1","maxFileSize":"1484","p75FileSize":"1484","p50FileSize":"1484","numAddedBytes":"1484"}	{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"1","executionTimeMs":"5499","numDeletionVectorsUpdated":"0","scanTimeMs":"2435","numAddedFiles":"1","numUpdatedRows":"1","numAddedBytes":"1720","rewriteTimeMs":"3062"}	{"numRemovedFiles":"1","numRemovedBytes":"1508","p25FileSize":"1498","numDeletionVectorsRemoved":"1","minFileSize":"1498","numAddedFiles":"1","maxFileSize":"1498","p75FileSize":"1498","p50FileSize":"1498","numAddedBytes":"1498"}	{"numRemovedFiles":"0","numRemovedBytes":"0","numCopiedRows":"0","numDeletionVectorsAdded":"1","numDeletionVectorsRemoved":"0","numAddedChangeFiles":"0","executionTimeMs":"3181","numDeletionVectorsUpdated":"0","numDeletedRows":"1","scanTimeMs":"2767","numAddedFiles":"0","numAddedBytes":"0","rewriteTimeMs":"414"}	{"numFiles":"1","numOutputRows":"5","numOutputBytes":"1508"}	{}
-- userMetadata	null	null	null	null	null	null
-- engineInfo	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13	Databricks-Runtime/18.2.x-photon-scala2.13

In [0]:
%sql
select * from table_changes('orders_ext', 1)
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,update_preimage,4,2026-06-06T04:36:47.023Z
-- 4,2013-07-25 00:00:00.0,8827,value,update_postimage,4,2026-06-06T04:36:47.023Z
-- 1,2013-07-25 00:00:00.0,11599,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,insert,1,2026-06-06T04:13:06.222Z
-- 5,2013-07-25 00:00:00.0,11318,COMPLETE,insert,1,2026-06-06T04:13:06.222Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,delete,2,2026-06-06T04:17:58.092Z


In [0]:
%sql
select * from table_changes('orders_ext', 2)
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,update_preimage,4,2026-06-06T04:36:47.023Z
-- 4,2013-07-25 00:00:00.0,8827,value,update_postimage,4,2026-06-06T04:36:47.023Z
-- 3,2013-07-25 00:00:00.0,12111,COMPLETE,delete,2,2026-06-06T04:17:58.092Z

In [0]:
%sql
select * from table_changes('orders_ext', 3)
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,update_preimage,4,2026-06-06T04:36:47.023Z
-- 4,2013-07-25 00:00:00.0,8827,value,update_postimage,4,2026-06-06T04:36:47.023Z

In [0]:
%sql
select * from table_changes('orders_ext', 4)
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- 4,2013-07-25 00:00:00.0,8827,CLOSED,update_preimage,4,2026-06-06T04:36:47.023Z
-- 4,2013-07-25 00:00:00.0,8827,value,update_postimage,4,2026-06-06T04:36:47.023Z


In [0]:
%sql
select * from table_changes('orders_ext', 5)
-- order_id,order_date,customer_id,order_status,_change_type,_commit_version,_commit_timestamp
-- No rows
